In [15]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import accuracy_score, r2_score, mean_squared_error
import numpy as np

# =========================
# 1. Load Data
# =========================
print("Loading data for helper models...")
try:
    data = pd.read_csv('housing.csv')
except FileNotFoundError:
    print("Error: 'housing.csv' not found. Please make sure the dataset is in the same directory.")
    exit()

# Drop rows with missing values
data.dropna(inplace=True)
print("Data loaded and cleaned.")

# =========================
# 2. Train Ocean Proximity Model (Helper Model 1)
# =========================
# This model is already highly accurate, so we'll keep it simple.
print("\nTraining ocean proximity prediction model...")
X_ocean = data[['latitude', 'longitude']]
y_ocean = data['ocean_proximity']

ocean_encoder = LabelEncoder()
y_ocean_encoded = ocean_encoder.fit_transform(y_ocean)

X_train_o, X_test_o, y_train_o, y_test_o = train_test_split(X_ocean, y_ocean_encoded, test_size=0.2, random_state=42)

ocean_model = KNeighborsClassifier(n_neighbors=5)
ocean_model.fit(X_train_o, y_train_o)

print("\nOcean Proximity Model Evaluation:")
print(f"Accuracy: {accuracy_score(ocean_model.predict(X_test_o), y_test_o):.4f}")

joblib.dump(ocean_model, 'ocean_model.joblib')
joblib.dump(ocean_encoder, 'ocean_encoder.joblib')
print("Ocean proximity model and encoder saved.")

# =========================
# 3. Optimize Random Forest for Income Level Model (Helper Model 2)
# =========================
print("\n--- Optimizing Random Forest for Best Income Level Model ---")
X_income = data[['latitude', 'longitude']]
y_income = data['median_income']

X_train_i, X_test_i, y_train_i, y_test_i = train_test_split(X_income, y_income, test_size=0.2, random_state=42)

# --- Define a more extensive hyperparameter grid for Random Forest ---
print("\nStarting extensive hyperparameter tuning for Random Forest...")
print("This may take several minutes as we are searching a large number of options...")
param_grid_rf = {
    'n_estimators': [200, 300, 400],
    'max_depth': [20, 30, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}

# --- Perform GridSearchCV ---
grid_search_rf = GridSearchCV(RandomForestRegressor(random_state=42), param_grid_rf, cv=3, n_jobs=-1, verbose=2, scoring='r2')
grid_search_rf.fit(X_train_i, y_train_i)

# --- Evaluate and Save the Best Model ---
print("\n--- Optimization Results ---")
print(f"Best Hyperparameters Found: {grid_search_rf.best_params_}")

best_rf_model = grid_search_rf.best_estimator_
y_pred_rf = best_rf_model.predict(X_test_i)
r2_rf = r2_score(y_test_i, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test_i, y_pred_rf))

print(f"\nOptimized Random Forest R² Score: {r2_rf:.4f}")
print(f"Optimized Random Forest RMSE: {rmse_rf:.2f}")

joblib.dump(best_rf_model, 'income_model.joblib')
print("\nOptimized income level model (Random Forest) saved as 'income_model.joblib'")

print("\nAll helper models have been successfully trained and saved!")



Loading data for helper models...
Data loaded and cleaned.

Training ocean proximity prediction model...

Ocean Proximity Model Evaluation:
Accuracy: 0.9883
Ocean proximity model and encoder saved.

--- Optimizing Random Forest for Best Income Level Model ---

Starting extensive hyperparameter tuning for Random Forest...
This may take several minutes as we are searching a large number of options...
Fitting 3 folds for each of 162 candidates, totalling 486 fits

--- Optimization Results ---
Best Hyperparameters Found: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 400}

Optimized Random Forest R² Score: 0.5779
Optimized Random Forest RMSE: 1.24

Optimized income level model (Random Forest) saved as 'income_model.joblib'

All helper models have been successfully trained and saved!
